# PHYT1D — Module 04: Complete Parameter Catalogue

Defines all four parameter groups with priors, bounds, and biological meanings.

| Group | Count | Source |
|-------|-------|--------|
| A — MCMC Window A | 8 | CGM + insulin + meal data |
| B — MCMC Window B | 6 | Exercise-window CGM |
| C — User-supplied | 14 | Runtime simulation inputs |
| D — Fixed population | 14 | Literature values |


In [ ]:
import numpy as np

# ═══════════════════════════════════════════════════════════════════
# GROUP A — Identified by MCMC Window A (from CGM + insulin + meal)
# ═══════════════════════════════════════════════════════════════════
GROUP_A_PARAMS = {
    "SI": {
        "unit"   : "mL/μU·min",
        "meaning": "Insulin sensitivity — rate insulin promotes glucose disposal",
        "prior"  : ("lognormal", -5.3, 0.6),      # LogN(mu, sigma) on log scale
        "bounds" : (1e-5, 0.1),
        "ref"    : "Bergman 1979",
    },
    "SG": {
        "unit"   : "min⁻¹",
        "meaning": "Glucose effectiveness — insulin-independent glucose disposal",
        "prior"  : ("normal", 0.025, 0.013),
        "bounds" : (0.001, 0.08),
        "ref"    : "Dalla Man 2007",
    },
    "Gb": {
        "unit"   : "mg/dL",
        "meaning": "Basal plasma glucose under optimal basal insulin",
        "prior"  : ("normal", 120.0, 10.0),
        "bounds" : (70.0, 200.0),
        "ref"    : "Dalla Man 2014",
    },
    "p2": {
        "unit"   : "min⁻¹",
        "meaning": "Insulin action rate constant — speed of X(t) response",
        "prior"  : ("normal", 0.05, 0.02),
        "bounds" : (0.001, 0.2),
        "ref"    : "Cappon 2023",
    },
    "kd": {
        "unit"   : "min⁻¹",
        "meaning": "Non-monomeric to monomeric insulin diffusion rate",
        "prior"  : ("normal", 0.026, 0.013),
        "bounds" : (0.001, 0.15),
        "ref"    : "Schiavon 2018",
    },
    "ka2": {
        "unit"   : "min⁻¹",
        "meaning": "Monomeric insulin SC absorption rate",
        "prior"  : ("normal", 0.066, 0.030),
        "bounds" : (0.001, 0.2),
        "ref"    : "Schiavon 2018",
    },
    "kempt": {
        "unit"   : "min⁻¹",
        "meaning": "Gastric emptying rate — speed CHO leaves stomach",
        "prior"  : ("normal", 0.055, 0.020),
        "bounds" : (0.005, 0.2),
        "ref"    : "Dalla Man 2006",
    },
    "kabs": {
        "unit"   : "min⁻¹",
        "meaning": "Intestinal glucose absorption rate — shapes Ra(t) peak",
        "prior"  : ("normal", 0.057, 0.020),
        "bounds" : (0.005, 0.2),
        "ref"    : "Dalla Man 2006",
    },
}

# ═══════════════════════════════════════════════════════════════════
# GROUP B — Identified by MCMC Window B (exercise-window CGM)
# ═══════════════════════════════════════════════════════════════════
GROUP_B_PARAMS = {
    "beta_aer": {
        "unit"   : "—",
        "meaning": "Aerobic SI gain coefficient per unit %VO2max",
        "prior"  : ("normal", 0.008, 0.003),
        "bounds" : (0.0, 0.05),
        "ref"    : "Riddell 2017",
    },
    "beta_res": {
        "unit"   : "—",
        "meaning": "Resistance EGP gain coefficient — catecholamine-driven hepatic surge",
        "prior"  : ("normal", 0.005, 0.002),
        "bounds" : (0.0, 0.03),
        "ref"    : "Yardley 2013",
    },
    "tau_post": {
        "unit"   : "min",
        "meaning": "Post-exercise SI decay time constant — KEY novel parameter",
        "prior"  : ("gamma", 4.0, 60.0),          # Gamma(shape=4, scale=60) → mean≈240
        "bounds" : (30.0, 600.0),
        "ref"    : "Hinshaw 2013",
    },
    "tau_on": {
        "unit"   : "min",
        "meaning": "SI ramp-up time constant during aerobic bout (~15 min to peak)",
        "prior"  : ("normal", 15.0, 5.0),
        "bounds" : (2.0, 45.0),
        "ref"    : "Riddell 2017",
    },
    "VO2max": {
        "unit"   : "mL/kg/min",
        "meaning": "Patient aerobic capacity — scaled by beta_aer and tau_post",
        "prior"  : ("normal", None, 8.0),          # mean from age/sex equation
        "bounds" : (15.0, 80.0),
        "ref"    : "Van de Poppe 2021",
    },
    "phi": {
        "unit"   : "—",
        "meaning": "Gastric emptying acceleration factor per unit exercise intensity",
        "prior"  : ("uniform", 0.0, 0.5),
        "bounds" : (0.0, 0.5),
        "ref"    : "Dalla Man 2006",
    },
}

# ═══════════════════════════════════════════════════════════════════
# GROUP C — User-supplied at simulation time
# ═══════════════════════════════════════════════════════════════════
def default_user_params():
    """
    Returns a dictionary of default Group C parameters (baseline scenario).
    All values can be overridden by the user before calling the simulator.
    """
    return {
        # Exercise prescription
        "u"            : 0.50,        # fractional VO2max intensity
        "d"            : 60.0,        # bout duration [min]
        "t_start"      : 420.0,       # exercise start = 14:00 (07:00 + 420 min)
        "exercise_type": "aerobic",   # 'aerobic' | 'resistance'

        # Insulin ratio
        "CR"           : 15.0,        # g/U carbohydrate-to-insulin ratio

        # Meal carbohydrate amounts [g]
        "CHO_BF"       : 45.0,
        "CHO_LU"       : 70.0,
        "CHO_DN"       : 60.0,
        "CHO_SN"       : 20.0,

        # Meal times [min from sim start; sim start = 07:00]
        "t_BF"         : 60.0,        # 08:00
        "t_LU"         : 300.0,       # 12:00
        "t_DN"         : 720.0,       # 19:00
        "t_SN"         : 960.0,       # 23:00

        # Modulation factors (nominal = 1.0)
        "bolus_factor" : 1.0,         # S3: multiply optimal bolus
        "CHO_factor"   : 1.0,         # S4: multiply all CHO amounts

        # Demography (for VO2max prior)
        "sex"          : "M",
        "age_group"    : "adult",
    }

# ═══════════════════════════════════════════════════════════════════
# GROUP D — Fixed population values
# ═══════════════════════════════════════════════════════════════════
GROUP_D_FIXED = {
    "VI"       : (0.126,  "L/kg",        "Volume of insulin distribution"),
    "ke"       : (0.127,  "min⁻¹",       "Insulin fractional clearance rate"),
    "beta"     : (8.0,    "min",         "SC absorption delay"),
    "f"        : (0.90,   "—",           "Fraction of CHO absorbed"),
    "VG"       : (1.45,   "dL/kg",       "Volume of glucose distribution"),
    "alpha"    : (7.0,    "min",         "Interstitial glucose delay constant"),
    "r1"       : (1.44,   "—",           "Hypoglycaemia risk shape 1"),
    "r2"       : (0.81,   "—",           "Hypoglycaemia risk shape 2"),
    "Gth"      : (60.0,   "mg/dL",       "Hypoglycaemia glucose threshold"),
    "Fc01"     : (0.0097, "mmol/kg/min", "Baseline non-insulin-mediated uptake"),
    "gamma_aer": (0.003,  "—",           "Aerobic muscle uptake coefficient"),
    "gamma_res": (0.0015, "—",           "Resistance muscle uptake coefficient"),
    "tau_ramp" : (20.0,   "min",         "Ramp-up time for muscle uptake / EGP"),
    "delta_res": (0.003,  "—",           "Resistance SI reduction coefficient"),
}

FIXED = {k: v[0] for k, v in GROUP_D_FIXED.items()}


## 4.1 Prior Sampling Utilities

In [ ]:
def sample_prior(param_name, param_dict, n=1, rng=None):
    """
    Draw n samples from the specified parameter prior.

    Parameters
    ----------
    param_name : str  — key in GROUP_A_PARAMS or GROUP_B_PARAMS
    param_dict : dict — one of GROUP_A_PARAMS / GROUP_B_PARAMS
    n          : int  — number of samples
    rng        : np.random.Generator | None

    Returns
    -------
    np.ndarray shape (n,)
    """
    if rng is None:
        rng = np.random.default_rng()
    prior = param_dict[param_name]["prior"]
    kind  = prior[0]

    if kind == "normal":
        _, mu, sigma = prior
        return rng.normal(mu, sigma, n)
    elif kind == "lognormal":
        _, mu_log, sigma_log = prior
        return np.exp(rng.normal(mu_log, sigma_log, n))
    elif kind == "gamma":
        _, shape, scale = prior
        return rng.gamma(shape, scale, n)
    elif kind == "uniform":
        _, lo, hi = prior
        return rng.uniform(lo, hi, n)
    else:
        raise ValueError(f"Unknown prior type: {kind}")


def default_theta_A():
    """Return prior means as a starting theta_A dict."""
    return {
        "SI"   : np.exp(-5.3),
        "SG"   : 0.025,
        "Gb"   : 120.0,
        "p2"   : 0.05,
        "kd"   : 0.026,
        "ka2"  : 0.066,
        "kempt": 0.055,
        "kabs" : 0.057,
    }


def default_theta_B(age_group="adult", sex="M"):
    """Return prior means as a starting theta_B dict."""
    from 03_vo2max_estimation import estimate_VO2max  # or inline below
    # Inline for portability:
    age_map = {"child": 10, "adolescent": 15, "adult": 35}
    age = age_map.get(age_group, 35)
    vo2 = (-0.0010*age**2 - 0.2248*age + 54.286 if sex == "M"
           else -0.0021*age**2 - 0.1407*age + 43.066)
    return {
        "beta_aer": 0.008,
        "beta_res": 0.005,
        "tau_post": 240.0,
        "tau_on"  : 15.0,
        "VO2max"  : vo2,
        "phi"     : 0.1,
    }


## 4.2 Parameter Summary Table

In [ ]:
import pandas as pd

def print_param_table(group_dict, group_label):
    rows = []
    for k, v in group_dict.items():
        prior = v["prior"]
        if prior[0] == "normal":
            prior_str = f"N({prior[1]}, {prior[2]}²)"
        elif prior[0] == "lognormal":
            prior_str = f"LogN({prior[1]}, {prior[2]}²)"
        elif prior[0] == "gamma":
            prior_str = f"Gamma({prior[1]}, {prior[2]}) → mean≈{prior[1]*prior[2]:.0f}"
        elif prior[0] == "uniform":
            prior_str = f"U({prior[1]}, {prior[2]})"
        else:
            prior_str = str(prior)
        rows.append({
            "Parameter": k,
            "Unit"     : v["unit"],
            "Prior"    : prior_str,
            "Ref"      : v["ref"],
        })
    df = pd.DataFrame(rows)
    print(f"\n{'='*60}")
    print(f"  Group {group_label} Parameters")
    print(f"{'='*60}")
    print(df.to_string(index=False))

print_param_table(GROUP_A_PARAMS, "A")
print_param_table(GROUP_B_PARAMS, "B")

print("\n── Group D Fixed Values ─────────────────────────────────────")
for k, (val, unit, desc) in GROUP_D_FIXED.items():
    print(f"  {k:12s}  {val:>8}  [{unit}]  — {desc}")
